In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
INPUT_PATH = Path("../data/results.csv")
OUTPUT_PATH = Path("../data/df_featured.csv")

In [3]:
def build_feature_state(df, base_elo=1500):
    state = {
        "elo": {},
        "matches_played": {},
        "win_history": {},
        "h2h": {},
    }
    teams = pd.concat([df["team_1"], df["team_2"]]).unique()
    for team in teams:
        state["elo"][team] = base_elo
        state["matches_played"][team] = 0
        state["win_history"][team] = []
    return state

In [4]:
def compute_features_for_match(match, state):
    team = match["team_1"]
    opp = match["team_2"]

    team_history = state["win_history"].get(team, [])
    opp_history = state["win_history"].get(opp, [])

    winrate_10 = np.mean(team_history[-10:]) if team_history else 0
    winrate_30 = np.mean(team_history[-30:]) if team_history else 0

    h2h_list = state["h2h"].get((team, opp), [])

    return {
        "elo_diff": state["elo"].get(team, 1500) - state["elo"].get(opp, 1500),
        "winrate_10_diff": winrate_10 - (np.mean(opp_history[-10:]) if opp_history else 0),
        "winrate_30_diff": winrate_30 - (np.mean(opp_history[-30:]) if opp_history else 0),
        "experience_diff": state["matches_played"].get(team, 0) - state["matches_played"].get(opp, 0),
        "rank_diff": match["rank_1"] - match["rank_2"],
        "h2h_winrate": np.mean(h2h_list) if h2h_list else 0.5,
    }

In [5]:
def update_state_with_result(match, state, k=32):
    team = match["team_1"]
    opp = match["team_2"]
    result = int(match["match_winner"] == 1)

    r_team = state["elo"].get(team, 1500)
    r_opp = state["elo"].get(opp, 1500)
    exp = 1 / (1 + 10 ** ((r_opp - r_team) / 400))
    state["elo"][team] = r_team + k * (result - exp)
    state["elo"][opp] = r_opp + k * ((1 - result) - (1 - exp))

    state["win_history"].setdefault(team, []).append(result)
    state["win_history"].setdefault(opp, []).append(1 - result)

    state["matches_played"][team] = state["matches_played"].get(team, 0) + 1
    state["matches_played"][opp] = state["matches_played"].get(opp, 0) + 1

    h2h_key = (team, opp)
    state["h2h"].setdefault(h2h_key, []).append(result)
    return state

In [6]:
def match_to_features(history_df, match):
    """Return raw (unscaled) feature row for a single future match given historical data."""
    past = history_df[history_df["date"] < match["date"]].sort_values("date")
    state = build_feature_state(past)
    for _, m in past.iterrows():
        update_state_with_result(m, state)
    features = compute_features_for_match(match, state)
    features["_map"] = match["_map"]
    return pd.DataFrame([features])

In [7]:
def build_features(df):
    state = build_feature_state(df)
    X, y = [], []
    for _, match in df.iterrows():
        X.append(compute_features_for_match(match, state))
        y.append(int(match["match_winner"] == 1))
        state = update_state_with_result(match, state)
    return pd.concat(
        [df, pd.DataFrame(X), pd.Series(y, name="team_1_wins")], axis=1
    )

In [8]:
df = pd.read_csv(INPUT_PATH, index_col=0, parse_dates=["date"])
print(f"Loaded {len(df)} rows from {INPUT_PATH}")

df_featured = build_features(df)

feature_cols = [
    "elo_diff", "winrate_10_diff", "winrate_30_diff",
    "experience_diff", "rank_diff", "h2h_winrate",
]

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_featured.to_csv(OUTPUT_PATH)
print(f"Saved {len(df_featured)} rows to {OUTPUT_PATH}")
df_featured[feature_cols + ["team_1_wins"]].describe()

Loaded 45773 rows from ../data/results.csv


Saved 45773 rows to ../data/df_featured.csv


,elo_diff,winrate_10_diff,winrate_30_diff,experience_diff,rank_diff,h2h_winrate,team_1_wins
count,45773.000000,45773.000000,45773.000000,45773.000000,45773.000000,45773.000000,45773.000000
mean,21.846916,0.029106,0.025384,18.416381,-8.895200,0.520559,0.535796
std,172.384724,0.375190,0.298048,221.978402,58.699372,0.345125,0.498722
min,-824.012874,-1.000000,-1.000000,-976.000000,-368.000000,0.000000,0.000000
25%,-86.622461,-0.200000,-0.166667,-66.000000,-32.000000,0.333333,0.000000
50%,19.463241,0.000000,0.033333,5.000000,-6.000000,0.500000,1.000000
75%,131.204925,0.300000,0.200000,102.000000,15.000000,0.833333,1.000000
max,823.458561,1.000000,1.000000,985.000000,368.000000,1.000000,1.000000
